In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
sns.set_theme(palette='pastel')

# Fragestellungen

1. In welchen Stadtteilen oder Regionen treten bestimmte Verbrechen häufiger auf?
    - Welche Stadtteile sind besonders von Drogenkriminalität betroffen?
    - Welche Art von Kriminalität hat in den Stadtteilen die letzten Jahren am stärksten zugenommen?
    - Gibt es saisonale unterschiede bei der Straftaten?
    - Gibt es eine Zunahme von Vandalismus während Feiertagen?
    - Sind Gewaltverbrechen häufiger an Wochenenden als an Wochentagen?
    - Hat die Corona-Pandemie die Anzahl bestimmter Verbrechen beeinflusst?
    - Welche Delikte werden häufiger in Verbindung mit anderen Straftaten begangen?

In [8]:
df = pd.read_csv('../data/oh_encoded_categories.csv', index_col=0)
df.head()

,date,title,location,link,details,number,details_lemma,category,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,sonstige,hasskriminalität,gewaltverbrechen
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841,Zusammenhang Oktober Jahr erfolgt Brandanschla...,vandalismus,False,True,False,False,False,False,False,False,False
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801,Nacht gemeinschaftlich begangen Raub jugendlic...,"gewaltverbrechen, diebstahl",False,False,False,False,False,True,False,False,True
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316,zwischenzeitlich Landeskriminalamt Rahmen inte...,betrug,False,False,False,False,True,False,False,False,False
3,2021-02-04 11:00:00,Gefährliche Körperverletzung,Mitte,/polizei/polizeimeldungen/pressemitteilung.960...,Die Kriminalpolizei der Direktion 2 bittet um ...,0276,Kriminalpolizei Direktion 2 bitten Mithilfe un...,gewaltverbrechen,False,False,False,False,False,False,False,False,True
4,2021-01-12 13:02:00,Verkehrsunfall mit schwerverletztem E-Bike-Fahrer,Mitte,/polizei/polizeimeldungen/pressemitteilung.103...,Gestern Nachmittag wurde in Mitte bei einem Ve...,0092,Gestern Nachmittag Mitte Verkehrsunfall E-Bike...,verkehrsdelikte,True,False,False,False,False,False,False,False,False


In [15]:
df = df[~df["location"].str.contains(r"[0-9]", regex=True)]

In [26]:
loc_df = df.groupby(by='location').agg(verkehrsdelikte=('verkehrsdelikte', 'sum'),
                                       vandalismus=('vandalismus', 'sum'),
                                       sexualdelikte=('sexualdelikte', 'sum'),
                                       drogen=('drogen', 'sum'),
                                       betrug=('betrug', 'sum'),
                                       diebstahl=('diebstahl', 'sum'),
                                       hasskriminalität=('hasskriminalität', 'sum'),
                                       gewaltverbrechen=('gewaltverbrechen', 'sum'),
                                       sonstige=('sonstige', 'sum')
                                       )
loc_df['gesamt'] = loc_df.sum(axis=1)
loc_df

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt
location,,,,,,,,,,
Charlottenburg-Wilmersdorf,315,154,7,36,2,183,69,231,252,1249
Friedrichshain-Kreuzberg,185,237,18,79,2,159,116,480,271,1547
Lichtenberg,171,181,9,22,1,80,60,240,155,919
Marzahn-Hellersdorf,174,135,4,15,2,76,53,181,151,791
Mitte,334,264,15,67,4,246,166,611,377,2084
Neukölln,211,205,7,63,6,160,82,370,204,1308
Pankow,250,153,4,27,0,119,70,208,189,1020
Reinickendorf,209,116,3,11,2,80,42,171,131,765
Spandau,221,128,2,29,5,89,41,174,148,837


In [33]:
import plotly.express as px

fig = px.bar(loc_df['gesamt'], 
             title="Anzahl der Verbrechen pro Stadtteil in Berlin", 
             labels={'gesamt': 'Anzahl der Verbrechen', 'Stadtteil': 'Stadtteil'}, color_continuous_scale='Viridis')

# Achsenbeschriftung und Layout anpassen
fig.update_layout(
    xaxis_title='Stadtteil',
    yaxis_title='Anzahl der Verbrechen',
    xaxis_tickangle=45,  
    margin=dict(l=40, r=40, t=40, b=60),  
    showlegend=False
)

fig.show()


In [37]:
einwohnerzahlen = {
    "Mitte": 397134,
    "Friedrichshain-Kreuzberg": 293454,
    "Pankow": 424307,
    "Charlottenburg-Wilmersdorf": 343081,
    "Spandau": 257091,
    "Steglitz-Zehlendorf": 310446,
    "Tempelhof-Schöneberg": 355868,
    "Neukölln": 330017,
    "Treptow-Köpenick": 294081,
    "Marzahn-Hellersdorf": 291948,
    "Lichtenberg": 311881,
    "Reinickendorf": 268792
}


loc_df['relative_sum'] = loc_df['gesamt'] / loc_df.index.map(einwohnerzahlen)

In [40]:
import plotly.express as px

fig = px.bar(loc_df['relative_sum'], 
             title="Anzahl der Verbrechen pro Stadtteil in Berlin", 
             labels={'gesamt': 'Anzahl der Verbrechen', 'Stadtteil': 'Stadtteil'}, color_continuous_scale='Viridis')

# Achsenbeschriftung und Layout anpassen
fig.update_layout(
    xaxis_title='Stadtteil',
    yaxis_title='Anzahl der Verbrechen',
    xaxis_tickangle=45,  
    margin=dict(l=40, r=40, t=40, b=60),  
    showlegend=False
)

fig.show()

In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Subplot mit zwei Y-Achsen
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Absolute Verbrechen (linke Y-Achse)
fig.add_trace(
    go.Bar(x=loc_df.index, y=loc_df['gesamt'], name='Absolute Verbrechen', marker_color='blue'),
    secondary_y=False
)

# Relative Verbrechen (rechte Y-Achse)
fig.add_trace(
    go.Scatter(x=loc_df.index, y=loc_df['relative_sum'], name='Relative Verbrechen pro Einwohner',
               mode='lines+markers', marker_color='green'),
    secondary_y=True
)

# Layout anpassen
fig.update_layout(
    title="Absolute und relative Verbrechen in Berliner Stadtteilen",
    xaxis_title="Stadtteil",
    yaxis_title="Anzahl der Verbrechen",
    yaxis2_title="Relative Verbrechen pro Einwohner",
    xaxis_tickangle=-45,
    height=600,
    width=1000
)

fig.show()


In [39]:
loc_df

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt,relative_sum
location,,,,,,,,,,,
Charlottenburg-Wilmersdorf,315,154,7,36,2,183,69,231,252,1249,0.003641
Friedrichshain-Kreuzberg,185,237,18,79,2,159,116,480,271,1547,0.005272
Lichtenberg,171,181,9,22,1,80,60,240,155,919,0.002947
Marzahn-Hellersdorf,174,135,4,15,2,76,53,181,151,791,0.002709
Mitte,334,264,15,67,4,246,166,611,377,2084,0.005248
Neukölln,211,205,7,63,6,160,82,370,204,1308,0.003963
Pankow,250,153,4,27,0,119,70,208,189,1020,0.002404
Reinickendorf,209,116,3,11,2,80,42,171,131,765,0.002846
Spandau,221,128,2,29,5,89,41,174,148,837,0.003256


In [54]:
loc_df.loc['all'] = df.sum(numeric_only=True)
loc_df

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt,relative_sum
location,,,,,,,,,,,
Charlottenburg-Wilmersdorf,315.0,154.0,7.0,36.0,2.0,183.0,69.0,231.0,252.0,1249.0,0.003641
Friedrichshain-Kreuzberg,185.0,237.0,18.0,79.0,2.0,159.0,116.0,480.0,271.0,1547.0,0.005272
Lichtenberg,171.0,181.0,9.0,22.0,1.0,80.0,60.0,240.0,155.0,919.0,0.002947
Marzahn-Hellersdorf,174.0,135.0,4.0,15.0,2.0,76.0,53.0,181.0,151.0,791.0,0.002709
Mitte,334.0,264.0,15.0,67.0,4.0,246.0,166.0,611.0,377.0,2084.0,0.005248
Neukölln,211.0,205.0,7.0,63.0,6.0,160.0,82.0,370.0,204.0,1308.0,0.003963
Pankow,250.0,153.0,4.0,27.0,0.0,119.0,70.0,208.0,189.0,1020.0,0.002404
Reinickendorf,209.0,116.0,3.0,11.0,2.0,80.0,42.0,171.0,131.0,765.0,0.002846
Spandau,221.0,128.0,2.0,29.0,5.0,89.0,41.0,174.0,148.0,837.0,0.003256


In [68]:
import plotly.express as px

# Die "all"-Zeile extrahieren
all_values = loc_df.loc['all'].dropna()  # Dropna entfernt die NaN-Werte (z. B. in 'relative_sum' und 'gesamt')

# Pie-Plot erstellen
fig = px.pie(names=all_values.index, values=all_values.values, title='Verteilung der Delikte in Berlin')
fig.show()

In [79]:
# Extrahiere die Bezirksnamen aus dem GeoJSON
geojson_bezirke = [feature['properties']['Gemeinde_name'] for feature in geojson_data['features']]

# Zeige die ersten paar Bezirksnamen an
print(geojson_bezirke[:12])  # Die ersten 10 Bezirksnamen aus dem GeoJSON


['Reinickendorf', 'Charlottenburg-Wilmersdorf', 'Treptow-Köpenick', 'Pankow', 'Neukölln', 'Lichtenberg', 'Marzahn-Hellersdorf', 'Spandau', 'Steglitz-Zehlendorf', 'Mitte', 'Friedrichshain-Kreuzberg', 'Tempelhof-Schöneberg']


In [91]:
loc_df.iloc[0:12]

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt,relative_sum
location,,,,,,,,,,,
Charlottenburg-Wilmersdorf,315.0,154.0,7.0,36.0,2.0,183.0,69.0,231.0,252.0,1249.0,0.003641
Friedrichshain-Kreuzberg,185.0,237.0,18.0,79.0,2.0,159.0,116.0,480.0,271.0,1547.0,0.005272
Lichtenberg,171.0,181.0,9.0,22.0,1.0,80.0,60.0,240.0,155.0,919.0,0.002947
Marzahn-Hellersdorf,174.0,135.0,4.0,15.0,2.0,76.0,53.0,181.0,151.0,791.0,0.002709
Mitte,334.0,264.0,15.0,67.0,4.0,246.0,166.0,611.0,377.0,2084.0,0.005248
Neukölln,211.0,205.0,7.0,63.0,6.0,160.0,82.0,370.0,204.0,1308.0,0.003963
Pankow,250.0,153.0,4.0,27.0,0.0,119.0,70.0,208.0,189.0,1020.0,0.002404
Reinickendorf,209.0,116.0,3.0,11.0,2.0,80.0,42.0,171.0,131.0,765.0,0.002846
Spandau,221.0,128.0,2.0,29.0,5.0,89.0,41.0,174.0,148.0,837.0,0.003256


In [81]:
loc_df.iloc[0:12]

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt,relative_sum
location,,,,,,,,,,,
Charlottenburg-Wilmersdorf,315.0,154.0,7.0,36.0,2.0,183.0,69.0,231.0,252.0,1249.0,0.003641
Friedrichshain-Kreuzberg,185.0,237.0,18.0,79.0,2.0,159.0,116.0,480.0,271.0,1547.0,0.005272
Lichtenberg,171.0,181.0,9.0,22.0,1.0,80.0,60.0,240.0,155.0,919.0,0.002947
Marzahn-Hellersdorf,174.0,135.0,4.0,15.0,2.0,76.0,53.0,181.0,151.0,791.0,0.002709
Mitte,334.0,264.0,15.0,67.0,4.0,246.0,166.0,611.0,377.0,2084.0,0.005248
Neukölln,211.0,205.0,7.0,63.0,6.0,160.0,82.0,370.0,204.0,1308.0,0.003963
Pankow,250.0,153.0,4.0,27.0,0.0,119.0,70.0,208.0,189.0,1020.0,0.002404
Reinickendorf,209.0,116.0,3.0,11.0,2.0,80.0,42.0,171.0,131.0,765.0,0.002846
Spandau,221.0,128.0,2.0,29.0,5.0,89.0,41.0,174.0,148.0,837.0,0.003256


In [95]:
fig = px.bar(loc_df.iloc[0:12], 
             x=loc_df.iloc[0:12].index,
             y=["gewaltverbrechen", "verkehrsdelikte", "diebstahl"],
             title="Verteilung der Straftaten nach Standort",
             labels={"value": "Anzahl der Straftaten", "location": "Ort"}
             )

fig.show()

In [98]:
df['date']

0      2021-04-17 13:00:00
1      2021-04-13 10:01:00
2      2021-02-09 15:01:00
3      2021-02-04 11:00:00
4      2021-01-12 13:02:00
              ...         
298    2025-01-01 12:03:00
299    2025-01-01 09:04:00
300    2024-10-25 11:02:00
301    2024-09-26 11:03:00
302    2024-05-01 14:00:00
Name: date, Length: 12144, dtype: object

In [96]:
df['date'] = pd.to_datetime(df['date'])
df["jahr"] = df["date"].dt.year
df_year = df.groupby(by=['location', "jahr"], as_index=False).agg(verkehrsdelikte=('verkehrsdelikte', 'sum'),
                                       vandalismus=('vandalismus', 'sum'),
                                       sexualdelikte=('sexualdelikte', 'sum'),
                                       drogen=('drogen', 'sum'),
                                       betrug=('betrug', 'sum'),
                                       diebstahl=('diebstahl', 'sum'),
                                       hasskriminalität=('hasskriminalität', 'sum'),
                                       gewaltverbrechen=('gewaltverbrechen', 'sum'),
                                       sonstige=('sonstige', 'sum')
                                       )
 
mitte_df_year = df_year[df_year["location"] == "Mitte"]
 
fig = px.bar(mitte_df_year,
             x="jahr",
             y=["verkehrsdelikte", "vandalismus", "diebstahl", "hasskriminalität", "gewaltverbrechen"],
             title="Verteilung der Straftaten nach Standort",
             labels={"value": "Anzahl der Straftaten", "jahr": "Jahre"},
             barmode="group"
             )
 
fig.show()

DateParseError: Unknown datetime string format, unable to parse: date, at position 0